# Segmentación Semántica y U-Net
## CC53 - Procesamiento de Imágenes

En este notebook exploraremos los conceptos fundamentales de la segmentación semántica y la arquitectura U-Net mediante ejemplos prácticos.

**Objetivos:**
- Entender qué es la segmentación semántica
- Conocer la arquitectura U-Net
- Implementar componentes básicos desde cero
- Evaluar resultados usando IoU

## 1. Importaciones y configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from scipy import ndimage

# Configurar semilla para reproducibilidad
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

# Configuración de matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 10

print("Librerías cargadas correctamente")
print(f"PyTorch versión: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. ¿Qué es la segmentación semántica?

La **segmentación semántica** es el proceso de etiquetar cada píxel de una imagen con una clase semántica.

**Diferencias con otras tareas:**
- **Clasificación:** Una etiqueta por imagen completa
- **Detección:** Etiqueta + caja delimitadora por objeto
- **Segmentación semántica:** Una etiqueta por píxel (misma clase = misma etiqueta)
- **Segmentación de instancias:** Una etiqueta por píxel (diferencia entre objetos de la misma clase)

Vamos a crear una imagen sintética simple para visualizar esto.

In [ ]:
def crear_imagen_sintetica(size=128, num_objetos=3, seed=SEED):
    """Crear una imagen sintética con objetos simples.
    
    Parámetros:
    - size: tamaño de la imagen (size x size)
    - num_objetos: número de formas geométricas a crear
    - seed: semilla para reproducibilidad
    
    Retorna:
    - imagen: array (size, size, 3) con valores [0, 255]
    - mask: array (size, size) con etiquetas de clases
    """
    np.random.seed(seed)
    
    # Crear imagen en blanco
    imagen = np.ones((size, size, 3), dtype=np.uint8) * 255
    mask = np.zeros((size, size), dtype=np.int32)  # 0 = fondo (blanco)
    
    # Crear objetos aleatorios
    for i in range(num_objetos):
        # Parámetros aleatorios
        y_center = np.random.randint(20, size - 20)
        x_center = np.random.randint(20, size - 20)
        radio = np.random.randint(10, 25)
        color_id = i + 1  # ID de clase (1, 2, 3, ...)
        
        # Crear máscara circular
        yy, xx = np.ogrid[:size, :size]
        dist = np.sqrt((yy - y_center)**2 + (xx - x_center)**2)
        circulo = dist <= radio
        
        # Color para este objeto
        colores = [
            [255, 0, 0],      # Rojo
            [0, 255, 0],      # Verde
            [0, 0, 255],      # Azul
            [255, 255, 0],    # Amarillo
        ]
        color = colores[i % len(colores)]
        
        # Aplicar al píxel
        imagen[circulo] = color
        mask[circulo] = color_id
    
    return imagen, mask

# Crear imagen de ejemplo
imagen_ejemplo, mask_ejemplo = crear_imagen_sintetica(size=128, num_objetos=3)

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Imagen original
axes[0].imshow(imagen_ejemplo)
axes[0].set_title('Imagen original')
axes[0].axis('off')

# Máscara de ground truth
cmap = ListedColormap(['white', 'red', 'green', 'blue'])
axes[1].imshow(mask_ejemplo, cmap=cmap, vmin=0, vmax=3)
axes[1].set_title('Máscara (Ground Truth)')
axes[1].axis('off')

# Visualizar clases
clases_info = f"Clases:\n0 = Fondo\n1 = Rojo\n2 = Verde\n3 = Azul"
axes[2].text(0.1, 0.5, clases_info, fontsize=12, family='monospace')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Imagen: {imagen_ejemplo.shape}")
print(f"Máscara: {mask_ejemplo.shape}")
print(f"Clases presentes: {np.unique(mask_ejemplo)}")

## 3. Métrica de evaluación: Intersection over Union (IoU)

IoU mide cuánto se superponen dos regiones:

$$IoU = \frac{\text{Área de solapamiento}}{\text{Área de unión}}$$

- **Rango:** 0 a 1
- **Interpretación:**
  - 0.0 a 0.3: Pobre
  - 0.3 a 0.7: Bueno
  - 0.7 a 1.0: Excelente

In [ ]:
def calcular_iou(prediccion, ground_truth, clase=None):
    """Calcular IoU entre predicción y ground truth.
    
    Parámetros:
    - prediccion: array con etiquetas predichas
    - ground_truth: array con etiquetas verdaderas
    - clase: si se especifica, calcular IoU solo para esa clase
             si es None, calcular IoU promedio sobre todas las clases
    
    Retorna:
    - iou: valor de IoU (0 a 1)
    """
    if clase is not None:
        # IoU para una clase específica
        pred_clase = (prediccion == clase).astype(np.float32)
        gt_clase = (ground_truth == clase).astype(np.float32)
        
        intersection = np.sum(pred_clase * gt_clase)
        union = np.sum((pred_clase + gt_clase) > 0)
        
        if union == 0:
            return 1.0 if intersection == 0 else 0.0
        
        return intersection / union
    else:
        # IoU promedio para todas las clases
        clases = np.unique(ground_truth)
        ious = []
        
        for c in clases:
            iou = calcular_iou(prediccion, ground_truth, clase=c)
            ious.append(iou)
        
        return np.mean(ious)

# Ejemplo: predicción perfecta
prediccion_perfecta = mask_ejemplo.copy()
iou_perfecto = calcular_iou(prediccion_perfecta, mask_ejemplo)
print(f"IoU predicción perfecta: {iou_perfecto:.4f}")

# Ejemplo: predicción con errores
prediccion_ruido = mask_ejemplo.copy()
# Añadir ruido aleatorio al 20% de los píxeles
mask_ruido = np.random.rand(*prediccion_ruido.shape) < 0.2
prediccion_ruido[mask_ruido] = np.random.randint(0, 4, np.sum(mask_ruido))
iou_ruido = calcular_iou(prediccion_ruido, mask_ejemplo)
print(f"IoU con ruido (20%): {iou_ruido:.4f}")

# Mostrar comparación visual
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(mask_ejemplo, cmap=cmap, vmin=0, vmax=3)
axes[0].set_title(f'Ground Truth (IoU = 1.0000)')
axes[0].axis('off')

axes[1].imshow(prediccion_ruido, cmap=cmap, vmin=0, vmax=3)
axes[1].set_title(f'Predicción con ruido (IoU = {iou_ruido:.4f})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Componentes básicos de U-Net

Ahora implementaremos los componentes clave de una red U-Net:
- **Conv 3×3 + ReLU:** Extrae características locales
- **Max Pool 2×2:** Reduce la dimensión espacial
- **Convolución transpuesta (Up-Conv) 2×2:** Aumenta la dimensión espacial
- **Conv 1×1:** Clasificación final por píxel

In [ ]:
class BloqueConvolutivo(nn.Module):
    """Bloque de dos convoluciones 3x3 con ReLU.
    
    Estructura:
    Conv3x3 -> ReLU -> Conv3x3 -> ReLU
    """
    def __init__(self, entrada_canales, salida_canales):
        super(BloqueConvolutivo, self).__init__()
        
        self.bloque = nn.Sequential(
            # Primera convolución
            nn.Conv2d(entrada_canales, salida_canales, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Segunda convolución
            nn.Conv2d(salida_canales, salida_canales, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.bloque(x)

# Probar el bloque
print("BloqueConvolutivo:")
bloque = BloqueConvolutivo(3, 64)
print(bloque)

# Probar con entrada de ejemplo
x_test = torch.randn(1, 3, 128, 128)  # (batch, canales, altura, ancho)
y_test = bloque(x_test)
print(f"Entrada: {x_test.shape}")
print(f"Salida: {y_test.shape}")

In [ ]:
class UNetSimple(nn.Module):
    """U-Net simplificada para segmentación.
    
    Arquitectura:
    Encoder:
      - Conv 3x3 (64) -> MaxPool 2x2
      - Conv 3x3 (128) -> MaxPool 2x2
      - Conv 3x3 (256) [cuello de botella]
    
    Decoder:
      - UpConv 2x2 (128) + skip -> Conv 3x3 (128)
      - UpConv 2x2 (64) + skip -> Conv 3x3 (64)
      - Conv 1x1 (num_clases) [clasificación final]
    """
    def __init__(self, entrada_canales=3, num_clases=4):
        super(UNetSimple, self).__init__()
        
        # ENCODER
        self.enc1 = BloqueConvolutivo(entrada_canales, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = BloqueConvolutivo(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        # CUELLO DE BOTELLA
        self.cuello = BloqueConvolutivo(128, 256)
        
        # DECODER
        self.upconv1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = BloqueConvolutivo(256, 128)  # 256 porque concatena con skip
        
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = BloqueConvolutivo(128, 64)   # 128 porque concatena con skip
        
        # CLASIFICACIÓN FINAL (Conv 1x1)
        self.final = nn.Conv2d(64, num_clases, kernel_size=1)
    
    def forward(self, x):
        # ENCODER con skip connections
        enc1_out = self.enc1(x)     # (batch, 64, H, W)
        pool1 = self.pool1(enc1_out)  # (batch, 64, H/2, W/2)
        
        enc2_out = self.enc2(pool1)   # (batch, 128, H/2, W/2)
        pool2 = self.pool2(enc2_out)  # (batch, 128, H/4, W/4)
        
        # CUELLO
        cuello = self.cuello(pool2)   # (batch, 256, H/4, W/4)
        
        # DECODER con concatenación (skip connections)
        upconv1 = self.upconv1(cuello)  # (batch, 128, H/2, W/2)
        # Concatenar con skip connection
        dec1_input = torch.cat([upconv1, enc2_out], dim=1)  # (batch, 256, H/2, W/2)
        dec1 = self.dec1(dec1_input)  # (batch, 128, H/2, W/2)
        
        upconv2 = self.upconv2(dec1)    # (batch, 64, H, W)
        # Concatenar con skip connection
        dec2_input = torch.cat([upconv2, enc1_out], dim=1)  # (batch, 128, H, W)
        dec2 = self.dec2(dec2_input)  # (batch, 64, H, W)
        
        # Clasificación final
        salida = self.final(dec2)  # (batch, num_clases, H, W)
        
        return salida

# Crear el modelo
modelo = UNetSimple(entrada_canales=3, num_clases=4)
print("\nU-Net Simplificada:")
print(modelo)

# Contar parámetros
total_params = sum(p.numel() for p in modelo.parameters())
trainables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"\nTotal de parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainables:,}")

In [ ]:
# Probar la red con una imagen de ejemplo
x_test = torch.randn(2, 3, 128, 128)  # 2 imágenes
y_pred = modelo(x_test)

print(f"Entrada: {x_test.shape}")
print(f"Salida (predicción): {y_pred.shape}")
print(f"\nInterpretación:")
print(f"  - Batch: {y_pred.shape[0]} imágenes")
print(f"  - Canales: {y_pred.shape[1]} clases (0=fondo, 1=rojo, 2=verde, 3=azul)")
print(f"  - Altura × Ancho: {y_pred.shape[2]} × {y_pred.shape[3]} (resolución original)")

## 5. Dataset sintético para entrenamiento

In [ ]:
class DatasetSegmentacion(Dataset):
    """Dataset de imágenes sintéticas para segmentación."""
    def __init__(self, num_muestras=100, size=128, seed=SEED):
        self.num_muestras = num_muestras
        self.size = size
        self.seed = seed
        
        # Generar imágenes de una vez
        np.random.seed(seed)
        self.imagenes = []
        self.mascaras = []
        
        for i in range(num_muestras):
            img, mask = crear_imagen_sintetica(size=size, num_objetos=3, seed=seed+i)
            self.imagenes.append(img)
            self.mascaras.append(mask)
    
    def __len__(self):
        return self.num_muestras
    
    def __getitem__(self, idx):
        # Convertir a tensores
        imagen = torch.from_numpy(self.imagenes[idx]).float() / 255.0
        imagen = imagen.permute(2, 0, 1)  # (H, W, 3) -> (3, H, W)
        
        mascara = torch.from_numpy(self.mascaras[idx]).long()
        
        return imagen, mascara

# Crear dataset
dataset_entrenamiento = DatasetSegmentacion(num_muestras=20, size=128)
dataset_validacion = DatasetSegmentacion(num_muestras=5, size=128, seed=SEED+1000)

print(f"Dataset de entrenamiento: {len(dataset_entrenamiento)} muestras")
print(f"Dataset de validación: {len(dataset_validacion)} muestras")

# Verificar una muestra
imagen_test, mascara_test = dataset_entrenamiento[0]
print(f"\nMuestra 0:")
print(f"  Imagen: {imagen_test.shape}")
print(f"  Máscara: {mascara_test.shape}")

## 6. Entrenamiento de la red

In [ ]:
# Configuración de entrenamiento
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10

# Crear dataloaders
dataloader_entrenamiento = DataLoader(
    dataset_entrenamiento, 
    batch_size=BATCH_SIZE, 
    shuffle=True
)
dataloader_validacion = DataLoader(
    dataset_validacion, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

# Crear modelo
modelo = UNetSimple(entrada_canales=3, num_clases=4).to(DEVICE)

# Función de pérdida y optimizador
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=LEARNING_RATE)

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Número de épocas: {NUM_EPOCHS}")

In [ ]:
def evaluar_modelo(modelo, dataloader, criterio, device):
    """Evaluar el modelo en el conjunto de validación."""
    modelo.eval()
    perdida_total = 0.0
    iou_total = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for imagenes, mascaras in dataloader:
            imagenes = imagenes.to(device)
            mascaras = mascaras.to(device)
            
            # Forward pass
            outputs = modelo(imagenes)
            perdida = criterio(outputs, mascaras)
            
            # Calcular IoU
            predicciones = torch.argmax(outputs, dim=1).cpu().numpy()
            mascaras_np = mascaras.cpu().numpy()
            
            for pred, gt in zip(predicciones, mascaras_np):
                iou = calcular_iou(pred, gt)
                iou_total += iou
            
            perdida_total += perdida.item()
            num_batches += 1
    
    modelo.train()
    return perdida_total / num_batches, iou_total / (num_batches * BATCH_SIZE)

# Entrenar
historico = {'perdida_entrenamiento': [], 'perdida_validacion': [], 'iou_validacion': []}

print("Entrenando...\n")
print(f"{'Época':<6} {'Pérdida Ent.':<15} {'Pérdida Val.':<15} {'IoU Val.':<10}")
print("-" * 50)

modelo.train()
for epoca in range(NUM_EPOCHS):
    perdida_epoca = 0.0
    num_batches = 0
    
    for imagenes, mascaras in dataloader_entrenamiento:
        imagenes = imagenes.to(DEVICE)
        mascaras = mascaras.to(DEVICE)
        
        # Forward pass
        outputs = modelo(imagenes)
        perdida = criterio(outputs, mascaras)
        
        # Backward pass
        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()
        
        perdida_epoca += perdida.item()
        num_batches += 1
    
    # Validación
    perdida_val, iou_val = evaluar_modelo(modelo, dataloader_validacion, criterio, DEVICE)
    
    # Guardar histórico
    perdida_ent_promedio = perdida_epoca / num_batches
    historico['perdida_entrenamiento'].append(perdida_ent_promedio)
    historico['perdida_validacion'].append(perdida_val)
    historico['iou_validacion'].append(iou_val)
    
    print(f"{epoca+1:<6} {perdida_ent_promedio:<15.4f} {perdida_val:<15.4f} {iou_val:<10.4f}")

print("\nEntrenamiento completado.")

## 7. Visualizar resultados del entrenamiento

In [ ]:
# Gráficas de pérdida e IoU
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pérdida
axes[0].plot(historico['perdida_entrenamiento'], label='Entrenamiento', marker='o')
axes[0].plot(historico['perdida_validacion'], label='Validación', marker='s')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida (Cross-Entropy)')
axes[0].set_title('Pérdida durante el entrenamiento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(historico['iou_validacion'], marker='o', color='green')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('IoU')
axes[1].set_title('IoU en validación')
axes[1].set_ylim([0, 1.0])
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Umbral: 0.5')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Pérdida inicial: {historico['perdida_entrenamiento'][0]:.4f}")
print(f"Pérdida final: {historico['perdida_entrenamiento'][-1]:.4f}")
print(f"IoU inicial: {historico['iou_validacion'][0]:.4f}")
print(f"IoU final: {historico['iou_validacion'][-1]:.4f}")

## 8. Predicción en nuevas imágenes

In [ ]:
# Crear una imagen de prueba nueva
imagen_prueba, mascara_verdadera = crear_imagen_sintetica(size=128, num_objetos=3, seed=SEED+5000)

# Convertir a tensor
imagen_tensor = torch.from_numpy(imagen_prueba).float() / 255.0
imagen_tensor = imagen_tensor.permute(2, 0, 1).unsqueeze(0)  # Añadir dimensión batch
imagen_tensor = imagen_tensor.to(DEVICE)

# Predicción
modelo.eval()
with torch.no_grad():
    salida = modelo(imagen_tensor)
    prediccion = torch.argmax(salida, dim=1).squeeze().cpu().numpy()

# Calcular IoU
iou_prueba = calcular_iou(prediccion, mascara_verdadera)

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(imagen_prueba)
axes[0].set_title('Imagen de entrada')
axes[0].axis('off')

axes[1].imshow(mascara_verdadera, cmap=cmap, vmin=0, vmax=3)
axes[1].set_title('Ground Truth')
axes[1].axis('off')

axes[2].imshow(prediccion, cmap=cmap, vmin=0, vmax=3)
axes[2].set_title(f'Predicción (IoU = {iou_prueba:.4f})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"IoU en imagen de prueba: {iou_prueba:.4f}")

## 9. Análisis por clase

In [ ]:
# Calcular IoU por clase
clases = ['Fondo', 'Rojo', 'Verde', 'Azul']
iou_por_clase = {}

for clase_id, clase_nombre in enumerate(clases):
    iou = calcular_iou(prediccion, mascara_verdadera, clase=clase_id)
    iou_por_clase[clase_nombre] = iou
    print(f"{clase_nombre:<10}: IoU = {iou:.4f}")

print(f"\nIoU promedio: {np.mean(list(iou_por_clase.values())):.4f}")

## 10. Conclusiones

### Conceptos clave aprendidos:

1. **Segmentación semántica:** Clasificación de cada píxel en clases semánticas

2. **Arquitectura U-Net:**
   - Encoder: comprime la información
   - Decoder: recupera la resolución
   - Skip connections: preservan detalles

3. **Componentes clave:**
   - Convoluciones 3×3: extraen características
   - Max pooling: reduce dimensión
   - Convoluciones transpuestas: aumentan dimensión
   - Convoluciones 1×1: clasificación final

4. **Métrica IoU:**
   - Mide solapamiento entre predicción y verdad
   - Rango 0 a 1
   - >0.7 se considera bueno

5. **Entrenamiento:**
   - Pérdida: Cross-entropy (por píxel)
   - Optimización: Adam
   - Evaluación: IoU en validación

### Aplicaciones prácticas:
- Conducción autónoma (segmentar carril, peatones, objetos)
- Imágenes médicas (segmentar tumores, órganos)
- Teledetección (clasificar terreno, cultivos)
- Edición de imágenes (cambiar fondo automáticamente)

### Próximos pasos:
- Entrenar con datasets reales (Cityscapes, COCO, etc.)
- Usar redes más profundas (ResNet, DenseNet como encoder)
- Implementar data augmentation
- Experimentar con diferentes funciones de pérdida
- Optimizar para inferencia en tiempo real